# Model 8: Temporal Fusion Transformer / Deep Attention Network for Drug `R03`

## Hyperparameter Selection Methodology:
Multi-head attention embed dimension ($d_{	ext{model}} = 32$) and heads ($H = 4$) evaluated on loss convergence.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'R03'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for R03 loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Deep Attention Model Execution for R03
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

scaler_mean = combined_series.mean()
scaler_std  = combined_series.std()
scaled_combined = (combined_series - scaler_mean) / scaler_std

seq_len = 28
X_seq, y_seq = [], []
for i in range(len(scaled_combined) - seq_len):
    X_seq.append(scaled_combined.values[i:i+seq_len])
    y_seq.append(scaled_combined.values[i+seq_len])

X_t = torch.tensor(np.array(X_seq), dtype=torch.float32).unsqueeze(-1)
y_t = torch.tensor(np.array(y_seq), dtype=torch.float32).unsqueeze(-1)

class DeepAttentionModel(nn.Module):
    def __init__(self, d_model=32):
        super(DeepAttentionModel, self).__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=4, batch_first=True)
        self.fc1 = nn.Linear(d_model, 16)
        self.fc2 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        h = self.input_proj(x)
        attn_out, _ = self.attn(h, h, h)
        out = self.relu(self.fc1(attn_out[:, -1, :]))
        out = self.fc2(out)
        return out

model_tft = DeepAttentionModel()
opt = torch.optim.Adam(model_tft.parameters(), lr=0.005)
crit = nn.MSELoss()

for epoch in range(40):
    opt.zero_grad()
    out = model_tft(X_t)
    loss = crit(out, y_t)
    loss.backward()
    opt.step()

model_tft.eval()
test_inputs = list(scaled_combined.values[-seq_len:])
tft_preds_scaled = []

for i in range(len(test_series)):
    x_in = torch.tensor(np.array(test_inputs[-seq_len:]), dtype=torch.float32).view(1, seq_len, 1)
    with torch.no_grad():
        p_val = model_tft(x_in).item()
    tft_preds_scaled.append(p_val)
    test_inputs.append((test_series.iloc[i] - scaler_mean) / scaler_std)

m8_test_pred = np.clip(np.array(tft_preds_scaled) * scaler_std + scaler_mean, 0, None)
test_metrics = evaluate_metrics(test_series, m8_test_pred)

print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 8: TFT / DEEP ATTENTION ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_TFT': m8_test_pred}).to_csv('m8_tft_preds.csv', index=False)


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 8: TFT / DEEP ATTENTION ===
  * RMSLE     : 1.0458
  * RMSE      : 7.7203
  * MAE       : 5.2949
  * WAPE (%)  : 76.4300
